In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet

future_dates = pd.date_range(
    start=clean_timeseries_df.index.max() + pd.Timedelta(days=1),
    periods=730, freq='D'
)
synthetic_future_df = pd.DataFrame(index=future_dates)
synthetic_future_df.index.name = 'Date'

for product in clean_timeseries_df.columns:
    print(f"Bootstrapping Years 2 & 3 for Product: {product}...")

    prophet_df = pd.DataFrame({
        'ds': clean_timeseries_df.index,
        'y': clean_timeseries_df[product].values
    })

    m = Prophet(daily_seasonality=False, yearly_seasonality=True, weekly_seasonality=True)
    m.fit(prophet_df)

    future = m.make_future_dataframe(periods=730)
    forecast = m.predict(future)

    year1_forecast = forecast['yhat'].iloc[:-730].values
    actuals = clean_timeseries_df[product].values
    residuals = actuals - year1_forecast

    future_baseline = forecast['yhat'].tail(730).values

    sampled_residuals = np.random.choice(residuals, size=730, replace=True)

    y_future_simulated = future_baseline + sampled_residuals

    y_future_cleaned = np.maximum(0, np.round(y_future_simulated))

    synthetic_future_df[product] = y_future_cleaned

print("\nSuccess! 2 Full Years of Synthetic Future Data Generated.")

In [ ]:
import plotly.graph_objects as go

for product in clean_timeseries_df.columns:
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=clean_timeseries_df.index,
        y=clean_timeseries_df[product],
        mode='lines',
        name='Year 1 (Actual 2019)',
        line=dict(color='blue')
    ))

    fig.add_trace(go.Scatter(
        x=synthetic_future_df.index,
        y=synthetic_future_df[product],
        mode='lines',
        name='Years 2 & 3 (Synthetic)',
        line=dict(color='orange')
    ))

    fig.update_layout(
        title=f'3-Year Digital Twin Validation: Product {product}',
        xaxis_title='Date',
        yaxis_title='Daily Quantity Sold',
        template='plotly_white',
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    fig.show()

In [ ]:
import pandas as pd

master_timeseries_df = pd.concat([clean_timeseries_df, synthetic_future_df])

master_timeseries_df = master_timeseries_df.sort_index()

print(f"Shape of the Master 3-Year Dataset: {master_timeseries_df.shape}")
print(f"Start Date: {master_timeseries_df.index.min().date()}")
print(f"End Date: {master_timeseries_df.index.max().date()}")

print("\nFirst 3 days (Actuals):")
print(master_timeseries_df.head(3))
print("\nLast 3 days (Synthetic):")
print(master_timeseries_df.tail(3))

In [ ]:
from google.colab import files

master_timeseries_df.to_csv('3yr_data.csv', index=True)

files.download('3yr_data.csv')

print("Download initiated!")